In [1]:
import sys; sys.path.append("..")

import joblib
import numpy as np
import pandas as pd

from src.config import ARTIFACTS, DATA_PROC, TEST_FRAC, VAL_FRAC
from src.data import (
    add_indices,
    aggregate_tags,
    build_index_maps,
    clean_movies,
    clean_ratings,
    load_raw,
    make_splits,
    to_csr,
)

ratings_raw, movies_raw, tags, _ = load_raw()
ratings = clean_ratings(ratings_raw)
movies = clean_movies(movies_raw)
movies = movies.merge(
    aggregate_tags(tags), left_on="movieId", right_index=True, how="left"
)
movies["tag_text"] = movies["tag_text"].fillna("")

user_ids, item_ids, user2idx, item2idx = build_index_maps(ratings, movies)
ratings = add_indices(ratings, user2idx, item2idx)
movies["i"] = movies["movieId"].map(item2idx).astype(np.int32)
movies = movies.sort_values("i").reset_index(drop=True)

train, val, test = make_splits(ratings)
n_users = len(user_ids)
n_items = len(item_ids)

print(f"Train: {len(train):,}; validation: {len(val):,}; test: {len(test):,}.")

user_indices = pd.RangeIndex(n_users)
user_bounds = pd.DataFrame(
    {
        "train_max": train.groupby("u")["timestamp"].max(),
        "val_min": val.groupby("u")["timestamp"].min(),
        "val_max": val.groupby("u")["timestamp"].max(),
        "test_min": test.groupby("u")["timestamp"].min(),
    }
).reindex(user_indices)
assert user_bounds.notna().all().all()
assert (user_bounds["train_max"] <= user_bounds["val_min"]).all()
assert (user_bounds["val_min"] <= user_bounds["val_max"]).all()
assert (user_bounds["val_max"] <= user_bounds["test_min"]).all()

train_pairs = set(map(tuple, train[["u", "i"]].to_numpy()))
val_pairs = set(map(tuple, val[["u", "i"]].to_numpy()))
test_pairs = set(map(tuple, test[["u", "i"]].to_numpy()))
assert train_pairs.isdisjoint(val_pairs)
assert train_pairs.isdisjoint(test_pairs)
assert val_pairs.isdisjoint(test_pairs)
assert len(train) + len(val) + len(test) == len(ratings)
assert np.array_equal(movies["i"].to_numpy(), np.arange(n_items))
assert set(train["u"].unique()) == set(user_indices)
assert set(val["u"].unique()) == set(user_indices)
assert set(test["u"].unique()) == set(user_indices)

train_csr = to_csr(train, n_users, n_items)
print(f"Train CSR shape: {train_csr.shape}; nnz: {train_csr.nnz:,}.")
train_item_counts = np.bincount(train["i"].to_numpy(), minlength=n_items)
cold_item_test_count = int((train_item_counts[test["i"].to_numpy()] == 0).sum())
print(f"Test ratings for items with zero train ratings: {cold_item_test_count:,}.")

DATA_PROC.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
train.to_parquet(DATA_PROC / "train.parquet", index=False)
val.to_parquet(DATA_PROC / "val.parquet", index=False)
test.to_parquet(DATA_PROC / "test.parquet", index=False)
movies.to_parquet(DATA_PROC / "movies.parquet", index=False)
joblib.dump(
    {
        "user_ids": user_ids,
        "item_ids": item_ids,
        "user2idx": user2idx,
        "item2idx": item2idx,
    },
    ARTIFACTS / "id_maps.joblib",
)

print(f"Saved processed data to {DATA_PROC} and ID maps to {ARTIFACTS / 'id_maps.joblib'}.")

Train: 72,115; validation: 8,304; test: 20,417.
Train CSR shape: (610, 9742); nnz: 72,115.
Test ratings for items with zero train ratings: 1,903.
Saved processed data to C:\Users\KRISHAN\Desktop\Fun\Projects\CineMatch_Recommendation_System\cinematch\data\processed and ID maps to C:\Users\KRISHAN\Desktop\Fun\Projects\CineMatch_Recommendation_System\cinematch\artifacts\id_maps.joblib.
